In [ ]:
import pandas as pd
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from itables import show
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm, ListedColormap, BoundaryNorm

In [ ]:
merged_df = pd.read_csv('data/merged_df.csv')

In [ ]:
merged_df['ND_GAIN_class'] = pd.cut(merged_df['ND_GAIN'],
                                    bins=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],
                                    labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                                    include_lowest=True)

In [ ]:
show(merged_df)

In [ ]:
merged_df_2010 = merged_df[merged_df['Year'] == 2010]


In [ ]:
# Map ISO3 codes to match world map dataset
# Create a copy to avoid modifying the original dataframe
merged_df_2010_mapped = merged_df_2010.copy()

# Define mappings: ND-GAIN ISO3 -> World Map ISO3
iso3_mappings = {
    'SCG': ['MNE', 'SRB'],  # Serbia and Montenegro -> Montenegro and Serbia
    'ISR': ['PSE'],  # Israel -> Palestine (additional mapping)
    'SDN': ['SSD'],  # Sudan -> South Sudan (additional mapping)
    'CHN': ['TWN']   # China -> Taiwan (additional mapping)
}

# For each mapping, duplicate rows with the new ISO3 codes
additional_rows = []
for source_iso3, target_iso3_list in iso3_mappings.items():
    source_data = merged_df_2010_mapped[merged_df_2010_mapped['ISO3'] == source_iso3]
    if not source_data.empty:
        for target_iso3 in target_iso3_list:
            new_rows = source_data.copy()
            new_rows['ISO3'] = target_iso3
            additional_rows.append(new_rows)

# Concatenate all additional rows to the dataframe
if additional_rows:
    merged_df_2010_mapped = pd.concat([merged_df_2010_mapped] + additional_rows, ignore_index=True)

print(f"Original rows: {len(merged_df_2010)}")
print(f"Rows after mapping: {len(merged_df_2010_mapped)}")

In [ ]:
show(merged_df_2010_mapped)

In [ ]:
# Load world geometry data directly from Natural Earth
url = 'https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip'
world = gpd.read_file(url)

# Use ISO_A3_EH instead of ISO_A3 (ISO_A3_EH has better coverage)
# Merge the world geometry with the MAPPED data based on ISO3 codes
world_merged = world.merge(merged_df_2010_mapped, left_on='ISO_A3_EH', right_on='ISO3', how='left')

# Create the plot
fig, ax = plt.subplots(1, 1, figsize=(15, 10))
world_merged.plot(column='ND_GAIN_class', 
                  ax=ax, 
                  legend=True,
                  cmap='RdYlGn',
                  edgecolor='black',
                  linewidth=0.5,
                  missing_kwds={'color': 'lightgrey'})
ax.set_title('ND-GAIN Class by Country (2010)', fontsize=16)
ax.axis('off')
plt.show()

In [ ]:
# Cross-check ISO3 codes between datasets
iso3_in_ndgain = set(merged_df_2010_mapped['ISO3'].dropna().unique())
iso3_in_world = set(world['ISO_A3_EH'].dropna().unique())

# Find codes in ND-GAIN but not in world map
missing_in_world = sorted(iso3_in_ndgain - iso3_in_world)

# Find codes in world map but not in ND-GAIN
missing_in_ndgain = sorted(iso3_in_world - iso3_in_ndgain)

print(f"Total ISO3 codes in ND-GAIN dataset: {len(iso3_in_ndgain)}")
print(f"Total ISO3 codes in World map dataset: {len(iso3_in_world)}")
print(f"\n{'='*60}")
print(f"ISO3 codes in ND-GAIN but NOT in World map ({len(missing_in_world)}):")
print(missing_in_world)
print(f"\n{'='*60}")
print(f"ISO3 codes in World map but NOT in ND-GAIN ({len(missing_in_ndgain)}):")
print(missing_in_ndgain)

## Coupling GHG, Vulnerability

Vulnerability wird noch umgepolt, damit es aufsteigt

In [ ]:
def add_deciles_quantile(dfy: pd.DataFrame, col: str, n_bins: int = 10) -> pd.Series:
    # Wandelt eine kontinuierliche Variable in Rang-basierte Quantile (Deciles) um
    # Wir arbeiten mit Rängen (nicht absoluten Werten), passend zu Spearman/Ranglogik
    # rank(method="first") verhindert Probleme bei vielen gleichen Werten
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


def rank_matrix_10x10(dfy: pd.DataFrame) -> pd.DataFrame:
    # Defensive Kopie, um das Original-DataFrame nicht zu verändern
    dfy = dfy.copy()

    # Emissions-Deciles (1 = niedrige Emissionen, 10 = hohe Emissionen)
    dfy["ghg_decile"] = add_deciles_quantile(dfy, "GHG_per_capita", 10)

    # Vulnerability-Deciles
    # WICHTIG: In deinem Datensatz ist Vulnerability bereits so definiert, dass
    # hohe Werte = hohe Vulnerabilität (z. B. Niger hoch, Schweiz tief).
    # Daher KEINE Inversion hier.
    dfy["vuln_decile"] = add_deciles_quantile(dfy, "Vulnerability", 10)

    # Erzeuge eine 10×10-Rangmatrix (Wert = Anzahl Länder)
    mat = (
        dfy.pivot_table(
            index="vuln_decile",
            columns="ghg_decile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 11), columns=range(1, 11), fill_value=0)
        .astype(int)
    )

    return mat


def plot_rank_matrix(mat: pd.DataFrame, year: int, cmap: str = "cividis") -> None:
    fig, ax = plt.subplots(figsize=(8.2, 6.2))

    # Wichtig gegen "weisses Grid" durch Rendering-Artefakte:
    # origin="lower" => y-Achse: 1 unten, 10 oben (passt zu Label 1=low, 10=high)
    im = ax.imshow(
        mat.values,
        cmap=cmap,
        aspect="auto",
        origin="lower",
        interpolation="none",
        resample=False
    )

    ax.set_title(f"10×10 Rank Matrix (Year {year}): Vulnerability × GHG per capita")
    ax.set_xlabel("GHG per capita decile (1 = low, 10 = high)")
    ax.set_ylabel("Vulnerability decile (1 = low, 10 = high)")

    ax.set_xticks(np.arange(10))
    ax.set_yticks(np.arange(10))
    ax.set_xticklabels(range(1, 11))
    ax.set_yticklabels(range(1, 11))

    # Kein Grid
    ax.grid(False)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Number of countries")

    # Textfarbe: weiss auf dunkel (kleine Werte), schwarz auf hell (grosse Werte)
    vmax = mat.values.max() if mat.values.size else 1
    threshold = 0.55 * vmax

    for i in range(10):
        for j in range(10):
            val = int(mat.iat[i, j])
            color = "white" if val <= threshold else "black"
            ax.text(j, i, str(val), ha="center", va="center", color=color, fontsize=9)

    plt.tight_layout()
    plt.show()


def run_rank_matrix(year: int | str = "latest", path: str = "data/merged_df.csv") -> pd.DataFrame:
    df = pd.read_csv(path)

    if year == "latest":
        year = int(df["Year"].max())
    else:
        year = int(year)

    dfy = df[df["Year"] == year].copy()

    mat = rank_matrix_10x10(dfy)
    plot_rank_matrix(mat, year=year, cmap="cividis")

    return mat


# --- Ausführung ---
mat = run_rank_matrix(year="latest", path="data/merged_df.csv")
mat


### Kontrolle Polaritäten Vulnerability über Extremwerte

In [ ]:
# Schau dir die Extremwerte an
# Datensatz explizit laden
df = pd.read_csv("data/merged_df.csv")

# Letztes Jahr bestimmen
latest_year = int(df["Year"].max())

# Auf letztes Jahr filtern
dfy = df[df["Year"] == latest_year].copy()

# Extremwerte anschauen
print("HÖCHSTE Vulnerability-Werte:")
display(
    dfy.sort_values("Vulnerability", ascending=False)
       [["Country", "ISO3", "Vulnerability"]]
       .head(15)
)

print("\nNIEDRIGSTE Vulnerability-Werte:")
display(
    dfy.sort_values("Vulnerability", ascending=True)
       [["Country", "ISO3", "Vulnerability"]]
       .head(15)
)



In [ ]:
def add_deciles_quantile(dfy: pd.DataFrame, col: str, n_bins: int = 10) -> pd.Series:
    # Rang-basierte Deciles, robust bei vielen gleichen Werten
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


# --- 1) Deciles + Kombi-Score auf deinem 2010-Frame erzeugen ---
# Annahme: merged_df_2010_mapped hat Spalten: ISO3, GHG_per_capita, Vulnerability
merged_df_2010_mapped = merged_df_2010_mapped.copy()

# GHG: 1 = low, 10 = high
merged_df_2010_mapped["ghg_decile"] = add_deciles_quantile(merged_df_2010_mapped, "GHG_per_capita", 10)

# Vulnerability: 1 = low, 10 = high (bei dir ist das bereits so orientiert!)
merged_df_2010_mapped["vuln_decile"] = add_deciles_quantile(merged_df_2010_mapped, "Vulnerability", 10)

# Kombi-Index:
# + hoch, wenn GHG hoch UND Vulnerability tief
# - tief, wenn GHG tief UND Vulnerability hoch
merged_df_2010_mapped["ghg_minus_vuln"] = merged_df_2010_mapped["ghg_decile"] - merged_df_2010_mapped["vuln_decile"]
# Wertebereich: -9 ... +9

# Optional: normalisieren auf [-1, +1] (praktisch für Legende/vergleich)
merged_df_2010_mapped["ghg_minus_vuln_norm"] = merged_df_2010_mapped["ghg_minus_vuln"] / 9.0

show(merged_df_2010_mapped)


# --- 2) Welt-Geometrie laden und mergen ---
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

world_merged = world.merge(
    merged_df_2010_mapped,
    left_on="ISO_A3_EH",
    right_on="ISO3",
    how="left"
)


# --- 3) Plot: Diverging Colormap (zwei Extreme) ---
fig, ax = plt.subplots(1, 1, figsize=(15, 10))

# Diverging cmap: negativ = forced-rider Extrem, positiv = free-rider Extrem
# (Du kannst auch "coolwarm" nehmen; das ist ebenfalls diverging.)
world_merged.plot(
    column="ghg_minus_vuln_norm",      # <- der neue Score
    ax=ax,
    legend=True,
    cmap="RdBu_r",                     # blau = negativ, rot = positiv (umgedreht im Zweifel)
    edgecolor="black",
    linewidth=0.5,
    missing_kwds={"color": "lightgrey"},
    vmin=-1, vmax=1                    # symmetrisch, damit 0 = neutral wirklich Mitte ist
)

ax.set_title(
    "2010: GHG (high) + Vulnerability (low)  ⟷  GHG (low) + Vulnerability (high)",
    fontsize=16
)
ax.axis("off")
plt.show()


In [ ]:
def add_quantile_bins(dfy: pd.DataFrame, col: str, n_bins: int = 5) -> pd.Series:
    # Rang-basierte Quantile (robust bei ties)
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


# --- 1) Quintile auf deinem 2010-Frame erzeugen ---
# Achsenlogik wie bei Althor:
#   x: Vulnerability (1 = low, 5 = high)
#   y: Emissions level (1 = low, 5 = high)
df5 = merged_df_2010_mapped.copy()

df5["ghg_q"] = add_quantile_bins(df5, "GHG_per_capita", 5)     # 1..5 (low..high)
df5["vuln_q"] = add_quantile_bins(df5, "Vulnerability", 5)     # 1..5 (low..high)

# Cell-ID: 25 Klassen
# Wir wollen eine eindeutige Klasse pro 5×5-Zelle.
# Reihenfolge (optional): y = ghg_q von low->high, x = vuln_q von low->high
# cell_id in [0..24] (praktisch für colormap)
df5["cell_id"] = (df5["ghg_q"] - 1) * 5 + (df5["vuln_q"] - 1)

# Optional: Für Tabelle sichtbar machen
show(df5[["Country","ISO3","Year","GHG_per_capita","Vulnerability","ghg_q","vuln_q","cell_id"]].head(30))


# --- 2) Althor-ähnliche 5×5 Farbcodes definieren ---
# Idee: Oben links (high GHG, low vuln) = "free rider" = rot/dunkelrot
#       Unten rechts (low GHG, high vuln) = "forced rider" = grün/dunkelgrün
#
# Achtung: Matplotlib colormap läuft von index 0..24.
# Unser cell_id ist so aufgebaut:
#   ghg_q steigt nach unten im Array (row-major), vuln_q nach rechts.
# Farbmatrix bauen wir aber bewusst als 5×5 "logical grid":
# rows = ghg_q low->high, cols = vuln_q low->high
#
# Du kannst die Farben später 1:1 feinjustieren.

# 5x5 Farben: rot -> gelb -> grün (Althor-like)
# row 0 = low emissions, row 4 = high emissions
# col 0 = low vulnerability, col 4 = high vulnerability
color_grid = np.array([
    ["#ffffcc", "#d9f0a3", "#addd8e", "#78c679", "#238443"],  # low ghg
    ["#ffeda0", "#fed976", "#feb24c", "#fd8d3c", "#31a354"],
    ["#feb24c", "#fd8d3c", "#fdae61", "#a6d96a", "#66bd63"],
    ["#fd8d3c", "#f03b20", "#fb6a4a", "#fdd49e", "#41ab5d"],
    ["#b10026", "#ef3b2c", "#fb6a4a", "#fdae61", "#ffff00"],  # high ghg (links rot)
], dtype=object)

# Hinweis:
# In Althor ist "free rider" oben links (high ghg, low vuln).
# Unser grid hat high ghg in der letzten Zeile (row 4), low vuln in col 0.
# Damit free rider "row 4, col 0" rot wird, muss die letzte Zeile links rot sein.
# forced rider (low ghg, high vuln) = row 0, col 4 = grün -> passt.

# In ListedColormap muss es eine flache Liste sein, in der Reihenfolge cell_id 0..24:
# cell_id = (ghg_q-1)*5 + (vuln_q-1) => row-major
cmap_althor = ListedColormap(color_grid.reshape(-1).tolist())


# --- 3) Welt-Geometrie laden und mergen ---
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

world_merged = world.merge(
    df5,
    left_on="ISO_A3_EH",
    right_on="ISO3",
    how="left"
)


# --- 4) Plot: diskrete 25 Klassen (5×5) statt Streifen ---
fig, ax = plt.subplots(1, 1, figsize=(15, 10))

world_merged.plot(
    column="cell_id",
    ax=ax,
    legend=True,
    cmap=cmap_althor,
    edgecolor="black",
    linewidth=0.4,
    missing_kwds={"color": "lightgrey"},
    vmin=0, vmax=24
)

ax.set_title("2010: 5×5 Matrix Classes (GHG per capita × Vulnerability) – Althor-style", fontsize=16)
ax.axis("off")
plt.show()


In [ ]:
def add_quantile_bins(dfy: pd.DataFrame, col: str, n_bins: int = 5) -> pd.Series:
    # Rang-basierte Quantile (robust bei ties)
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


def make_5x5_cmap(base_cmap_name: str = "cividis") -> tuple[ListedColormap, np.ndarray]:
    """
    Erzeugt eine diskrete 25-Farben-Map aus einem farbenblind-tauglichen Continuous-Cmap.
    Gleichzeitig wird eine 5×5-Farbmatrix zurückgegeben (für die inset-Legende).
    """
    base = plt.get_cmap(base_cmap_name)

    # 25 diskrete Farben gleichmässig aus der Continuous-Cmap sampeln
    colors = [base(x) for x in np.linspace(0.05, 0.95, 25)]
    cmap25 = ListedColormap(colors)

    # In 5×5 Rasterform bringen (row-major)
    color_grid = np.arange(25).reshape(5, 5)  # Werte 0..24, nur fuer Legendenbild
    return cmap25, color_grid


# --- 1) Quintile + 25 Klassen (wie Althor: 5×5) ---
df5 = merged_df_2010_mapped.copy()

# Achsenlogik wie bei Althor:
# x: Vulnerability (1 = low, 5 = high)
# y: Emissions level (1 = low, 5 = high)
df5["ghg_q"] = add_quantile_bins(df5, "GHG_per_capita", 5)   # 1..5
df5["vuln_q"] = add_quantile_bins(df5, "Vulnerability", 5)   # 1..5

# cell_id: 0..24 (row-major)
# row = ghg_q-1 (low->high), col = vuln_q-1 (low->high)
df5["cell_id"] = (df5["ghg_q"] - 1) * 5 + (df5["vuln_q"] - 1)


# --- 2) Welt-Geometrie mergen ---
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

world_merged = world.merge(
    df5,
    left_on="ISO_A3_EH",
    right_on="ISO3",
    how="left"
)


# --- 3) Colormap: farbenblind-tauglich (Blau–Gelb) ---
# Empfehlung: "cividis" (sehr gut für Farbblindheit), alternativ: "viridis"
cmap25, legend_grid = make_5x5_cmap(base_cmap_name="cividis")


# --- 4) Plot: Karte OHNE Streifen-Colorbar + Inset 5×5 Farbmatrix ---
fig, ax = plt.subplots(1, 1, figsize=(15, 9))

world_merged.plot(
    column="cell_id",
    ax=ax,
    cmap=cmap25,
    edgecolor="black",
    linewidth=0.4,
    missing_kwds={"color": "lightgrey"}
)

ax.set_title("2010: 5×5 Matrix Classes (GHG per capita × Vulnerability) – Althor-style", fontsize=16)
ax.axis("off")


# --- 5) Inset: 5×5 Farbmatrix-Legende (wie im Paper) ---
# Inset-Achse platzieren (links unten wie Althor; bei Bedarf bbox_to_anchor anpassen)
ax_leg = inset_axes(
    ax,
    width="22%", height="35%",
    loc="lower left",
    bbox_to_anchor=(0.05, 0.08, 1, 1),
    bbox_transform=ax.transAxes,
    borderpad=0
)

# 5×5 Matrix anzeigen (origin="lower" => unten = low emissions, oben = high emissions)
im_leg = ax_leg.imshow(
    legend_grid,
    cmap=cmap25,
    origin="lower",
    interpolation="none"
)

# Achsenticks/Labels wie Althor
ax_leg.set_xticks(range(5))
ax_leg.set_yticks(range(5))
ax_leg.set_xticklabels([1, 2, 3, 4, 5])
ax_leg.set_yticklabels([1, 2, 3, 4, 5])

ax_leg.set_xlabel("Vulnerability level\n(1 = low → 5 = high)", fontsize=9)
ax_leg.set_ylabel("Emissions level\n(1 = low → 5 = high)", fontsize=9)

# Optional: Ecken labels (Free/Forced) wie im Paper
# Althor: Free rider = high emissions + low vulnerability => (y=5, x=1) => top-left
# Forced rider = low emissions + high vulnerability => (y=1, x=5) => bottom-right
ax_leg.text(0, 4, "Free\nrider", ha="center", va="center", fontsize=8, fontweight="bold")
ax_leg.text(4, 0, "Forced\nrider", ha="center", va="center", fontsize=8, fontweight="bold")

# Rahmen klar machen
for spine in ax_leg.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()


In [ ]:
def add_deciles_quantile(dfy: pd.DataFrame, col: str, n_bins: int = 10) -> pd.Series:
    # Rang-basierte Deciles (robust bei ties)
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


def compute_deciles(dfy: pd.DataFrame) -> pd.DataFrame:
    """
    Erzeugt ghg_decile und vuln_decile (je 1..10, low..high) und gibt dfy zurück.
    Annahme: Vulnerability ist bereits so orientiert, dass hohe Werte = hohe Vulnerabilität.
    """
    dfy = dfy.copy()
    dfy["ghg_decile"] = add_deciles_quantile(dfy, "GHG_per_capita", 10)
    dfy["vuln_decile"] = add_deciles_quantile(dfy, "Vulnerability", 10)
    return dfy


def build_count_matrix_10x10(dfy: pd.DataFrame) -> pd.DataFrame:
    """
    10×10 Matrix: Zeilen = Vulnerability-Deciles (1..10), Spalten = GHG-Deciles (1..10),
    Werte = Anzahl Länder (nunique ISO3).
    """
    mat = (
        dfy.pivot_table(
            index="vuln_decile",
            columns="ghg_decile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 11), columns=range(1, 11), fill_value=0)
        .astype(int)
    )
    return mat


def build_equity_color_matrix_10x10() -> np.ndarray:
    """
    Baut eine 10×10 Farbwert-Matrix im Althor-Sinn, sodass:
    - alle Zellen auf der Diagonale (y=x) denselben Farbw ert haben (Equity-Band)
    - je weiter weg von der Diagonale, desto "extremer" die Farbe

    Wir codieren Farbe über d = (vuln_decile - ghg_decile).
    d < 0 => high GHG & low vuln (Free-rider Richtung)
    d > 0 => low GHG & high vuln (Forced-rider Richtung)
    """
    # d liegt in [-9, +9]
    grid = np.zeros((10, 10), dtype=int)
    for i in range(10):       # i = vuln index 0..9
        for j in range(10):   # j = ghg  index 0..9
            d = (i + 1) - (j + 1)
            grid[i, j] = d
    return grid


def plot_rank_matrix_equity_style(
    counts: pd.DataFrame,
    year: int,
    cmap: str = "cividis",
    show_zeros: bool = True
) -> None:
    """
    Plottet die 10×10 Zählmatrix, aber färbt Zellen nach Distanz zur Diagonale (Equity-Stil),
    sodass die Diagonale überall gleich gefärbt ist.
    Entfernt die weissen Linien (kein Grid, kein Interpolations-Artefakt).
    """
    # Farbwerte: Distanz zur Diagonale als "Ungerechtigkeits-Intensität"
    # dist = |vuln - ghg| => 0..9, Diagonale immer 0 (gleiche Farbe)
    dist = np.abs(build_equity_color_matrix_10x10())  # 10×10, 0..9

    fig, ax = plt.subplots(figsize=(8.4, 6.6))

    # WICHTIG gegen weisse Linien:
    # - interpolation="none"
    # - resample=False
    # - aspect="equal" (quadratische Pixel reduzieren Render-Artefakte)
    # - origin="lower" damit 1 unten, 10 oben
    im = ax.imshow(
        dist,
        cmap=cmap,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal",
        vmin=0,
        vmax=9
    )

    ax.set_title(f"10×10 Equity-Style Matrix (Year {year}): Vulnerability × GHG per capita")
    ax.set_xlabel("GHG per capita decile (1 = low, 10 = high)")
    ax.set_ylabel("Vulnerability decile (1 = low, 10 = high)")

    ax.set_xticks(np.arange(10))
    ax.set_yticks(np.arange(10))
    ax.set_xticklabels(range(1, 11))
    ax.set_yticklabels(range(1, 11))

    # Keine Gitterlinien / keine minor ticks
    ax.grid(False)
    ax.minorticks_off()

    # Colorbar beschreibt jetzt "distance from equity diagonal"
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("|Vulnerability decile − GHG decile| (0 = equitable)")

    # Zellzahlen = Anzahl Länder in dieser (vuln, ghg)-Zelle
    vmax_counts = counts.values.max() if counts.values.size else 1
    threshold = 0.55 * vmax_counts

    for i in range(10):
        for j in range(10):
            val = int(counts.iat[i, j])

            if (not show_zeros) and val == 0:
                continue

            # Textfarbe auf Basis des Hintergrunds (dist): dunkel => weiss, hell => schwarz
            # cividis: low dist eher dunkel -> weiss; high dist eher hell -> schwarz
            # (Wenn du eine andere cmap nimmst und es dreht: einfach Bedingung invertieren.)
            color = "white" if dist[i, j] <= 4 else "black"

            ax.text(j, i, str(val), ha="center", va="center", color=color, fontsize=9)

    plt.tight_layout()
    plt.show()


def run_equity_matrix(
    year: int | str = "latest",
    path: str = "data/merged_df.csv",
    cmap: str = "cividis",
) -> pd.DataFrame:
    df = pd.read_csv(path)

    if year == "latest":
        year = int(df["Year"].max())
    else:
        year = int(year)

    dfy = df[df["Year"] == year].copy()
    dfy = compute_deciles(dfy)

    counts = build_count_matrix_10x10(dfy)
    plot_rank_matrix_equity_style(counts, year=year, cmap=cmap, show_zeros=True)

    return counts


# --- Ausführung ---
counts = run_equity_matrix(year="latest", path="data/merged_df.csv", cmap="cividis")
counts


In [ ]:
def add_deciles_quantile(dfy: pd.DataFrame, col: str, n_bins: int = 10) -> pd.Series:
    # Rang-basierte Deciles (robust bei ties)
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


def compute_deciles(dfy: pd.DataFrame) -> pd.DataFrame:
    """
    Erzeugt ghg_decile und vuln_decile (je 1..10, low..high).
    Annahme: Vulnerability ist bereits so orientiert, dass hohe Werte = hohe Vulnerabilität.
    """
    dfy = dfy.copy()
    dfy["ghg_decile"] = add_deciles_quantile(dfy, "GHG_per_capita", 10)   # 1..10
    dfy["vuln_decile"] = add_deciles_quantile(dfy, "Vulnerability", 10)   # 1..10
    return dfy


def build_count_matrix_10x10(dfy: pd.DataFrame) -> pd.DataFrame:
    """
    10×10 Matrix: Zeilen = Vulnerability-Deciles (1..10), Spalten = GHG-Deciles (1..10),
    Werte = Anzahl Länder (nunique ISO3).
    """
    mat = (
        dfy.pivot_table(
            index="vuln_decile",
            columns="ghg_decile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 11), columns=range(1, 11), fill_value=0)
        .astype(int)
    )
    return mat


def equity_distance_grid_10x10() -> np.ndarray:
    """
    dist = |vuln_decile - ghg_decile| in [0..9]
    -> Diagonale überall 0 => gleiche Farbe (Equity-Band)
    """
    grid = np.zeros((10, 10), dtype=int)
    for i in range(10):       # i: vuln index 0..9
        for j in range(10):   # j: ghg  index 0..9
            grid[i, j] = abs((i + 1) - (j + 1))
    return grid


def discrete_cmap_from_continuous(base_cmap: str = "cividis", n: int = 10) -> ListedColormap:
    """
    Baut eine diskrete Colormap mit n Stufen aus einer Continuous-Colormap.
    Für dist=0..9 brauchen wir n=10.
    """
    base = plt.get_cmap(base_cmap)
    colors = [base(x) for x in np.linspace(0.05, 0.95, n)]
    return ListedColormap(colors)


def plot_map_and_matrix_10x10_equity_style(
    df_2010: pd.DataFrame,
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
    cmap_name: str = "cividis"
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:
    """
    Produziert eine Figure mit:
    - links: Weltkarte nach Equity-Distanz (0..9)
    - rechts: 10×10 Rank-Matrix (Counts) mit gleichem Farbschema (Diagonale gleich)
    """
    dfy = compute_deciles(df_2010)

    # Distanz zur Equity-Diagonale als gemeinsame "Klasse" (0..9)
    dfy["equity_dist"] = (dfy["vuln_decile"] - dfy["ghg_decile"]).abs().astype(int)

    # 10×10 Count-Matrix (für Zahlenanzeige)
    counts = build_count_matrix_10x10(dfy)

    # 10×10 Farbgrid (für Matrix-Färbung)
    dist_grid = equity_distance_grid_10x10()

    # diskrete cmap mit 10 Stufen: 0..9
    cmap10 = discrete_cmap_from_continuous(cmap_name, n=10)

    # World geometry
    world = gpd.read_file(world_url)
    world_merged = world.merge(
        dfy,
        left_on="ISO_A3_EH",
        right_on="ISO3",
        how="left"
    )

    # --- Figure: 1×2 Layout ---
    fig, (ax_map, ax_mat) = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.8, 1]})

    # --- A) MAP (keine Colorbar-Streifen: wir machen eine diskrete Legende über Colorbar,
    # aber mit klaren 10 Stufen; wenn du gar keine willst, kann man sie ausblenden)
    world_merged.plot(
        column="equity_dist",
        ax=ax_map,
        cmap=cmap10,
        edgecolor="black",
        linewidth=0.35,
        missing_kwds={"color": "lightgrey"},
        vmin=0, vmax=9,
        legend=True,
        legend_kwds={"label": "|Vulnerability decile − GHG decile| (0 = equitable)"}
    )
    ax_map.set_title("2010: Equity-style classes (Diagonale = gleiche Farbe)", fontsize=14)
    ax_map.axis("off")

    # --- B) MATRIX (gleiches Farbschema, keine weissen Linien) ---
    im = ax_mat.imshow(
        dist_grid,
        cmap=cmap10,
        origin="lower",
        interpolation="none",   # verhindert weisse Kanten durch Interpolation
        resample=False,         # verhindert Resampling-Artefakte
        aspect="equal",
        vmin=0, vmax=9
    )

    ax_mat.set_title("10×10 Rank Matrix: Vulnerability × GHG", fontsize=14)
    ax_mat.set_xlabel("GHG per capita decile (1 = low, 10 = high)")
    ax_mat.set_ylabel("Vulnerability decile (1 = low, 10 = high)")
    ax_mat.set_xticks(np.arange(10))
    ax_mat.set_yticks(np.arange(10))
    ax_mat.set_xticklabels(range(1, 11))
    ax_mat.set_yticklabels(range(1, 11))

    # Kein Grid, keine minor ticks => keine weissen Linien
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    # Zahlen (Counts) in die Matrix schreiben
    for i in range(10):
        for j in range(10):
            val = int(counts.iat[i, j])
            # Textfarbe nach Hintergrundklasse:
            # dist klein (0..4) eher dunkel -> weiss; dist gross (5..9) eher hell -> schwarz
            color = "white" if dist_grid[i, j] <= 4 else "black"
            ax_mat.text(j, i, str(val), ha="center", va="center", color=color, fontsize=9)

    plt.tight_layout()
    plt.show()

    return counts, world_merged


# --- RUN ---
# Erwartet: merged_df_2010_mapped enthält ISO3, GHG_per_capita, Vulnerability, Year, Country
counts_10x10, world_merged_2010 = plot_map_and_matrix_10x10_equity_style(
    df_2010=merged_df_2010_mapped,
    cmap_name="cividis"
)

counts_10x10


In [ ]:

def add_deciles_quantile(dfy: pd.DataFrame, col: str, n_bins: int = 10) -> pd.Series:
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


def compute_deciles(dfy: pd.DataFrame) -> pd.DataFrame:
    """
    Annahme: Vulnerability ist so orientiert, dass hohe Werte = hohe Vulnerabilität.
    """
    dfy = dfy.copy()
    dfy["ghg_decile"] = add_deciles_quantile(dfy, "GHG_per_capita", 10)   # 1..10
    dfy["vuln_decile"] = add_deciles_quantile(dfy, "Vulnerability", 10)   # 1..10
    return dfy


def build_count_matrix_10x10(dfy: pd.DataFrame) -> pd.DataFrame:
    mat = (
        dfy.pivot_table(
            index="vuln_decile",
            columns="ghg_decile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 11), columns=range(1, 11), fill_value=0)
        .astype(int)
    )
    return mat


def build_class_grid_10x10(band: int = 1) -> np.ndarray:
    """
    Liefert ein 10×10 Grid mit Klassen:
      0 = forced rider (vuln >> ghg)  -> d = vuln - ghg > band
      1 = equitable (nahe Diagonale)  -> |d| <= band
      2 = free rider (ghg >> vuln)    -> d = vuln - ghg < -band
    """
    grid = np.zeros((10, 10), dtype=int)
    for i in range(10):       # vuln index
        for j in range(10):   # ghg index
            d = (i + 1) - (j + 1)  # vuln - ghg
            if abs(d) <= band:
                grid[i, j] = 1  # equitable
            elif d > band:
                grid[i, j] = 0  # forced
            else:
                grid[i, j] = 2  # free
    return grid


def classify_country_rows(dfy: pd.DataFrame, band: int = 1) -> pd.DataFrame:
    """
    Klassifiziert jedes Land in forced/equitable/free basierend auf d = vuln_decile - ghg_decile.
    Output-Spalte: equity_class (0/1/2 wie oben)
    """
    dfy = dfy.copy()
    d = (dfy["vuln_decile"] - dfy["ghg_decile"]).astype(int)

    dfy["equity_class"] = np.where(
        d > band, 0,
        np.where(d < -band, 2, 1)
    ).astype(int)

    return dfy


def plot_map_and_matrix_10x10_3class(
    df_2010: pd.DataFrame,
    year: int = 2010,
    band: int = 1,
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:

    # --- Prep ---
    dfy = compute_deciles(df_2010)
    dfy = classify_country_rows(dfy, band=band)
    counts = build_count_matrix_10x10(dfy)

    class_grid = build_class_grid_10x10(band=band)

    # --- Colorblind 3-class palette (Blau – Hellgrau – Orange) ---
    # forced = blue, equitable = light grey, free = orange
    cmap3 = ListedColormap(["#2C7BB6", "#F2F2F2", "#FDAE61"])
    norm3 = BoundaryNorm(boundaries=[-0.5, 0.5, 1.5, 2.5], ncolors=3)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Plot layout ---
    fig, (ax_map, ax_mat) = plt.subplots(
        1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.8, 1]}
    )

    # --- A) Map (3 categories) ---
    world_merged.plot(
        column="equity_class",
        ax=ax_map,
        cmap=cmap3,
        norm=norm3,
        edgecolor="black",
        linewidth=0.35,
        missing_kwds={"color": "lightgrey"},
        legend=False
    )
    ax_map.set_title(f"{year}: Forced / Equitable / Free (band = ±{band} decile)", fontsize=14)
    ax_map.axis("off")

    # Manuelle Legend (klarer als GeoPandas colorbar)
    legend_labels = ["Forced riders", "Equitable", "Free riders"]
    legend_handles = [
        plt.Line2D([0], [0], marker="s", linestyle="", markersize=12,
                   markerfacecolor=cmap3(i), markeredgecolor="black")
        for i in range(3)
    ]
    ax_map.legend(legend_handles, legend_labels, loc="lower left", frameon=True)

    # --- B) Matrix (gleiches 3-class Schema) ---
    im = ax_mat.imshow(
        class_grid,
        cmap=cmap3,
        norm=norm3,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal"
    )

    ax_mat.set_title("10×10 Matrix (3-class Althor effect)", fontsize=14)
    ax_mat.set_xlabel("GHG per capita decile (1 = low, 10 = high)")
    ax_mat.set_ylabel("Vulnerability decile (1 = low, 10 = high)")
    ax_mat.set_xticks(np.arange(10))
    ax_mat.set_yticks(np.arange(10))
    ax_mat.set_xticklabels(range(1, 11))
    ax_mat.set_yticklabels(range(1, 11))
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    # Zellzahlen (Counts) einzeichnen
    # Textfarbe: auf blau/orange weiss, auf grau schwarz
    for i in range(10):
        for j in range(10):
            val = int(counts.iat[i, j])
            cls = class_grid[i, j]
            txt_color = "black" if cls == 1 else "white"
            ax_mat.text(j, i, str(val), ha="center", va="center", color=txt_color, fontsize=9)

    # Optional: Labels wie im Paper
    # Free rider = high GHG (rechts) + low vuln (unten) => bottom-right Bereich
    # Forced rider = low GHG (links) + high vuln (oben) => top-left Bereich
    ax_mat.text(0.3, 9.4, "Forced\nriders", ha="left", va="top", fontsize=9, color="black")
    ax_mat.text(9.6, 0.3, "Free\nriders", ha="right", va="bottom", fontsize=9, color="black")
    ax_mat.text(5.0, 5.0, "Equitable", ha="center", va="center", fontsize=10, color="black")

    plt.tight_layout()
    plt.show()

    return counts, world_merged


# --- RUN ---
# band:
#   0 = nur exakte Diagonale ist equitable
#   1 = Diagonale ±1 ist equitable (empfohlen, wirkt wie "Band" im Paper)
counts_10x10, world_merged_2010 = plot_map_and_matrix_10x10_3class(
    df_2010=merged_df_2010_mapped,
    year=2010,
    band=1
)

counts_10x10


In [ ]:
def add_deciles_quantile(dfy: pd.DataFrame, col: str, n_bins: int = 10) -> pd.Series:
    return pd.qcut(
        dfy[col].rank(method="first"),
        q=n_bins,
        labels=list(range(1, n_bins + 1))
    ).astype(int)


def compute_deciles(dfy: pd.DataFrame) -> pd.DataFrame:
    dfy = dfy.copy()
    dfy["ghg_decile"] = add_deciles_quantile(dfy, "GHG_per_capita", 10)   # 1..10
    dfy["vuln_decile"] = add_deciles_quantile(dfy, "Vulnerability", 10)   # 1..10
    return dfy


def build_count_matrix_10x10(dfy: pd.DataFrame) -> pd.DataFrame:
    return (
        dfy.pivot_table(
            index="vuln_decile",
            columns="ghg_decile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 11), columns=range(1, 11), fill_value=0)
        .astype(int)
    )


def make_blue_grey_yellow_cmap() -> LinearSegmentedColormap:
    # farbblind-tauglich, klarer Kontrast: Blau -> Grau -> Gelb
    # (du kannst die 3 Farben jederzeit fein-tunen)
    return LinearSegmentedColormap.from_list(
        "blue_grey_yellow",
        ["#2C7BB6", "#F2F2F2", "#FEE08B"],  # blue, grey, yellow
        N=256
    )


def add_equity_score(dfy: pd.DataFrame, band: int = 1) -> pd.DataFrame:
    """
    score = ghg_decile - vuln_decile
      score > 0  -> Free-rider Richtung (hohe GHG, tiefere Vulnerability)
      score < 0  -> Forced-rider Richtung (tiefe GHG, höhere Vulnerability)
      |score| <= band -> Equitable-Band (score = 0)
    """
    dfy = dfy.copy()
    diff = (dfy["ghg_decile"] - dfy["vuln_decile"]).astype(int)
    dfy["equity_score"] = np.where(np.abs(diff) <= band, 0, diff).astype(int)
    return dfy


def build_score_grid_10x10(band: int = 1) -> np.ndarray:
    """
    Liefert ein 10x10 Grid mit equity_score pro Zelle (nicht counts!),
    damit die inset-Matrix exakt die gleiche Farb-Logik wie die Karte hat.
    """
    grid = np.zeros((10, 10), dtype=int)
    for i in range(10):       # vuln decile = i+1 (y)
        for j in range(10):   # ghg  decile = j+1 (x)
            diff = (j + 1) - (i + 1)  # ghg - vuln
            grid[i, j] = 0 if abs(diff) <= band else diff
    return grid


def plot_map_and_matrix_10x10_equity_gradient(
    df_2010: pd.DataFrame,
    year: int = 2010,
    band: int = 1,
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:

    # --- Prep ---
    dfy = compute_deciles(df_2010)
    dfy = add_equity_score(dfy, band=band)

    counts = build_count_matrix_10x10(dfy)
    score_grid = build_score_grid_10x10(band=band)

    # max mögliche Distanz in 10 Deciles: 9
    vmax = 9
    vmin = -9

    cmap = make_blue_grey_yellow_cmap()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Layout ---
    fig, (ax_map, ax_mat) = plt.subplots(
        1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.8, 1]}
    )

    # --- A) Map (equity_score) ---
    world_merged.plot(
        column="equity_score",
        ax=ax_map,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.35,
        missing_kwds={"color": "lightgrey"},
        legend=False
    )
    ax_map.set_title(f"{year}: Equity gradient (band = ±{band} = grau)", fontsize=14)
    ax_map.axis("off")

    # Colorbar (für Richtung + Stärke)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_map, fraction=0.035, pad=0.02)
    cbar.set_label("equity_score = GHG decile − Vulnerability decile\n(0 = equitable band)", fontsize=10)

    # --- B) 10×10 Matrix: Farbe = equity_score, Zahl = count ---
    im = ax_mat.imshow(
        score_grid,
        cmap=cmap,
        norm=norm,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal"
    )

    ax_mat.set_title("10×10 Matrix: Farbe = Equity, Zahl = # Länder", fontsize=14)
    ax_mat.set_xlabel("GHG per capita decile (1 = low, 10 = high)")
    ax_mat.set_ylabel("Vulnerability decile (1 = low, 10 = high)")
    ax_mat.set_xticks(np.arange(10))
    ax_mat.set_yticks(np.arange(10))
    ax_mat.set_xticklabels(range(1, 11))
    ax_mat.set_yticklabels(range(1, 11))
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    # Zellzahlen (Counts) einzeichnen
    # Textfarbe passend zum Hintergrund:
    # - im grauen Band: schwarz
    # - sonst: weiss, wenn |score| gross (dunkler), sonst schwarz (heller)
    for i in range(10):
        for j in range(10):
            val = int(counts.iat[i, j])
            sc = int(score_grid[i, j])

            if sc == 0:
                txt_color = "black"
            else:
                # einfache Heuristik: weiter weg von 0 -> eher weiss
                txt_color = "white" if abs(sc) >= 5 else "black"

            ax_mat.text(j, i, str(val), ha="center", va="center", color=txt_color, fontsize=9)

    # Orientierungstext (wie Paper-Feeling)
    ax_mat.text(0.3, 9.6, "Forced\nrider side", ha="left", va="top", fontsize=9, color="black")
    ax_mat.text(9.6, 0.3, "Free\nrider side", ha="right", va="bottom", fontsize=9, color="black")
    ax_mat.text(5.0, 5.0, "Equitable band", ha="center", va="center", fontsize=10, color="black")

    plt.tight_layout()
    plt.show()

    return counts, world_merged


# --- RUN ---
# band=1 oder band=2 gibt meist den “breiten grauen Streifen” Effekt wie du ihn magst.
counts_10x10, world_merged_2010 = plot_map_and_matrix_10x10_equity_gradient(
    df_2010=merged_df_2010_mapped,
    year=2010,
    band=2
)

counts_10x10


In [ ]:
def add_deciles_fixed_bounds(dfy: pd.DataFrame, col: str, vmin: float, vmax: float, n_bins: int = 10) -> pd.Series:
    """
    Erzeugt Deciles mit FESTEN Grenzen (nicht Quantile).
    Teilt den Bereich [vmin, vmax] in n_bins gleiche Intervalle.
    """
    bins = np.linspace(vmin, vmax, n_bins + 1)
    return pd.cut(
        dfy[col],
        bins=bins,
        labels=list(range(1, n_bins + 1)),
        include_lowest=True
    ).astype(int)


def compute_deciles_fixed_bounds(dfy: pd.DataFrame, ghg_max: float) -> pd.DataFrame:
    """
    GHG: feste Grenzen von 0 bis ghg_max
    Vulnerability: feste Grenzen von 0 bis 1
    """
    dfy = dfy.copy()
    dfy["ghg_decile"] = add_deciles_fixed_bounds(dfy, "GHG_per_capita", vmin=0, vmax=ghg_max, n_bins=10)
    dfy["vuln_decile"] = add_deciles_fixed_bounds(dfy, "Vulnerability", vmin=0, vmax=1, n_bins=10)
    return dfy


def build_count_matrix_10x10(dfy: pd.DataFrame) -> pd.DataFrame:
    return (
        dfy.pivot_table(
            index="vuln_decile",
            columns="ghg_decile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 11), columns=range(1, 11), fill_value=0)
        .astype(int)
    )


def make_blue_grey_yellow_cmap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list(
        "blue_grey_yellow",
        ["#2C7BB6", "#F2F2F2", "#FEE08B"],
        N=256
    )


def add_equity_score(dfy: pd.DataFrame, band: int = 1) -> pd.DataFrame:
    """
    score = ghg_decile - vuln_decile
      score > 0  -> Free-rider Richtung (hohe GHG, tiefere Vulnerability)
      score < 0  -> Forced-rider Richtung (tiefe GHG, höhere Vulnerability)
      |score| <= band -> Equitable-Band (score = 0)
    """
    dfy = dfy.copy()
    diff = (dfy["ghg_decile"] - dfy["vuln_decile"]).astype(int)
    dfy["equity_score"] = np.where(np.abs(diff) <= band, 0, diff).astype(int)
    return dfy


def build_score_grid_10x10(band: int = 1) -> np.ndarray:
    """
    Liefert ein 10x10 Grid mit equity_score pro Zelle.
    """
    grid = np.zeros((10, 10), dtype=int)
    for i in range(10):       # vuln decile = i+1 (y)
        for j in range(10):   # ghg  decile = j+1 (x)
            diff = (j + 1) - (i + 1)  # ghg - vuln
            grid[i, j] = 0 if abs(diff) <= band else diff
    return grid


def plot_map_and_matrix_10x10_equity_fixed_bounds(
    df_2010: pd.DataFrame,
    year: int = 2010,
    band: int = 1,
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:

    # --- Bestimme max GHG über ALLE Jahre (oder nur dieses Jahr) ---
    # Hier nehmen wir nur das aktuelle Jahr
    ghg_max = df_2010["GHG_per_capita"].max()
    print(f"GHG max für {year}: {ghg_max:.2f}")
    print(f"Vulnerability Grenzen: 0.0 - 1.0")
    print(f"Equitable band: ±{band} Diagonale(n)\n")

    # --- Prep mit festen Grenzen ---
    dfy = compute_deciles_fixed_bounds(df_2010, ghg_max=ghg_max)
    dfy = add_equity_score(dfy, band=band)

    counts = build_count_matrix_10x10(dfy)
    score_grid = build_score_grid_10x10(band=band)

    vmax = 9
    vmin = -9

    cmap = make_blue_grey_yellow_cmap()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Layout ---
    fig, (ax_map, ax_mat) = plt.subplots(
        1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.8, 1]}
    )

    # --- A) Map (equity_score) ---
    world_merged.plot(
        column="equity_score",
        ax=ax_map,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.35,
        missing_kwds={"color": "lightgrey"},
        legend=False
    )
    ax_map.set_title(
        f"{year}: Equity gradient (band = ±{band}, GHG: 0-{ghg_max:.1f}, Vuln: 0-1)", 
        fontsize=14
    )
    ax_map.axis("off")

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_map, fraction=0.035, pad=0.02)
    cbar.set_label("equity_score = GHG decile − Vulnerability decile\n(0 = equitable band)", fontsize=10)

    # --- B) 10×10 Matrix: Farbe = equity_score, Zahl = count ---
    im = ax_mat.imshow(
        score_grid,
        cmap=cmap,
        norm=norm,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal"
    )

    ax_mat.set_title("10×10 Matrix: Farbe = Equity, Zahl = # Länder", fontsize=14)
    ax_mat.set_xlabel("GHG per capita decile (1 = low, 10 = high)")
    ax_mat.set_ylabel("Vulnerability decile (1 = low, 10 = high)")
    ax_mat.set_xticks(np.arange(10))
    ax_mat.set_yticks(np.arange(10))
    ax_mat.set_xticklabels(range(1, 11))
    ax_mat.set_yticklabels(range(1, 11))
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    # Zellzahlen (Counts) einzeichnen
    for i in range(10):
        for j in range(10):
            val = int(counts.iat[i, j])
            sc = int(score_grid[i, j])

            if sc == 0:
                txt_color = "black"
            else:
                txt_color = "white" if abs(sc) >= 5 else "black"

            ax_mat.text(j, i, str(val), ha="center", va="center", color=txt_color, fontsize=9)

    # Orientierungstext
    ax_mat.text(0.3, 9.6, "Forced\nrider side", ha="left", va="top", fontsize=9, color="black")
    ax_mat.text(9.6, 0.3, "Free\nrider side", ha="right", va="bottom", fontsize=9, color="black")
    ax_mat.text(5.0, 5.0, "Equitable band", ha="center", va="center", fontsize=10, color="black")

    plt.tight_layout()
    plt.show()

    return counts, world_merged


# --- RUN mit festen Grenzen und band=1 (nur 3 Diagonalen: -1, 0, +1) ---
counts_10x10_fixed, world_merged_2010_fixed = plot_map_and_matrix_10x10_equity_fixed_bounds(
    df_2010=merged_df_2010_mapped,
    year=2010,
    band=1
)

counts_10x10_fixed

In [ ]:
def add_quintiles_by_rank_qcut(dfy: pd.DataFrame, col: str, n_bins: int = 5) -> pd.Series:
    """
    Paper-style Quintiles: split countries into n_bins equally-sized groups (by rank).
    Robust gegen Ties/identische Werte: qcut auf Rängen (rank(method='first')).
    
    Ergebnis: 1 = low, ..., 5 = high ("acute emissions" im Paper).
    """
    s = dfy[col]
    out = pd.Series(pd.NA, index=dfy.index, dtype="Int64")

    mask = s.notna()
    if mask.sum() == 0:
        return out

    # Rank macht Werte eindeutig -> qcut bricht nicht bei vielen Duplikaten.
    r = s[mask].rank(method="first", ascending=True)
    q = pd.qcut(r, q=n_bins, labels=list(range(1, n_bins + 1)))

    out.loc[mask] = q.astype(int)
    return out


# -----------------------------
# Vulnerability handling (Paper has categories low/medium/high/severe/acute)
# -----------------------------
VULN_CATEGORY_MAP = {
    "low": 1,
    "medium": 2,
    "moderate": 2,   # falls deine Daten "moderate" verwenden
    "high": 3,
    "severe": 4,
    "acute": 5
}

def to_vuln_ordinal_1to5(
    dfy: pd.DataFrame,
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_0to1: bool = True  # NEU: Fixed bounds für 0-1 Skala
) -> pd.Series:
    """
    - Wenn vuln_category_col existiert (Strings wie low/medium/...): map auf 1..5
    - Wenn vuln_col bereits 1..5 numerisch: verwende direkt
    - Wenn use_fixed_bounds_0to1=True und vuln_col in [0,1]: 
      Fixed bounds in 0.2er Schritten (wissenschaftlich korrekt für normierte Skalen!)
        1 = [0.0 - 0.2)   low
        2 = [0.2 - 0.4)   medium
        3 = [0.4 - 0.6)   high
        4 = [0.6 - 0.8)   severe
        5 = [0.8 - 1.0]   acute
    """
    if vuln_category_col and vuln_category_col in dfy.columns:
        s = dfy[vuln_category_col].astype(str).str.strip().str.lower()
        mapped = s.map(VULN_CATEGORY_MAP)
        return mapped.astype("Int64")

    s = dfy[vuln_col]
    # Wenn schon 1..5 (oder subset) -> direkt
    if pd.api.types.is_numeric_dtype(s):
        uniq = pd.Series(s.dropna().unique())
        if len(uniq) > 0 and set(uniq.astype(int).tolist()).issubset({1, 2, 3, 4, 5}):
            return s.astype("Int64")

    # NEU: Fixed bounds für normierte 0-1 Skala (wissenschaftlich korrekt!)
    if use_fixed_bounds_0to1 and pd.api.types.is_numeric_dtype(s):
        # Fixed bounds: 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
        bins = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
        labels = [1, 2, 3, 4, 5]
        
        out = pd.Series(pd.NA, index=dfy.index, dtype="Int64")
        mask = s.notna()
        
        if mask.sum() > 0:
            # pd.cut mit include_lowest=True damit 0.0 auch in Bin 1 fällt
            cut_result = pd.cut(s[mask], bins=bins, labels=labels, include_lowest=True)
            out.loc[mask] = cut_result.astype(int)
        
        return out

    # sonst: NA
    return pd.Series(pd.NA, index=dfy.index, dtype="Int64")


def compute_bins_paper_style(
    dfy: pd.DataFrame,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_vuln: bool = True  # NEU: Fixed bounds für Vulnerability (0-1)
) -> pd.DataFrame:
    """
    Paper-konform:
      - GHG: Quintile (Quantile-Bins) pro dfy -> relative Verteilung
      - Vulnerability: Fixed bounds in 0.2er-Schritten (0-1) -> absolute Vergleichbarkeit
        WICHTIG: Dies ist wissenschaftlich korrekt für normierte Skalen!
    """
    dfy = dfy.copy()
    dfy["ghg_quintile"] = add_quintiles_by_rank_qcut(dfy, ghg_col, n_bins=5)
    dfy["vuln_quintile"] = to_vuln_ordinal_1to5(
        dfy,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_0to1=use_fixed_bounds_vuln  # Fixed bounds für 0-1 Skala
    )
    return dfy


def build_count_matrix_5x5(dfy: pd.DataFrame) -> pd.DataFrame:
    return (
        dfy.pivot_table(
            index="vuln_quintile",
            columns="ghg_quintile",
            values="ISO3",
            aggfunc="nunique",
            fill_value=0
        )
        .reindex(index=range(1, 6), columns=range(1, 6), fill_value=0)
        .astype(int)
    )


def make_blue_grey_yellow_cmap() -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list(
        "blue_grey_yellow",
        ["#2C7BB6", "#F2F2F2", "#FEE08B"],
        N=256
    )


def add_equity_score_5x5(dfy: pd.DataFrame, band: int = 0) -> pd.DataFrame:
    """
    score = ghg_quintile - vuln_quintile
      score > 0  -> Free-rider Richtung
      score < 0  -> Forced-rider Richtung
      |score| <= band -> Equitable-Band
    """
    dfy = dfy.copy()
    diff = (dfy["ghg_quintile"] - dfy["vuln_quintile"])
    
    # Wichtig: Konvertiere zu float für np.where (kann mit NaN umgehen)
    diff_float = diff.astype(float)
    
    # np.where mit float-Werten (NaN bleibt NaN)
    equity_score_float = np.where(np.abs(diff_float) <= band, 0, diff_float)
    
    # Zurück zu Int64 (NaN wird zu pd.NA)
    dfy["equity_score"] = pd.array(equity_score_float, dtype="Int64")
    
    return dfy


def build_score_grid_5x5(band: int = 0) -> np.ndarray:
    grid = np.zeros((5, 5), dtype=int)
    for i in range(5):       # vuln = i+1
        for j in range(5):   # ghg  = j+1
            diff = (j + 1) - (i + 1)
            grid[i, j] = 0 if abs(diff) <= band else diff
    return grid


def plot_map_and_matrix_5x5_equity_paper_style(
    df_year: pd.DataFrame,
    year: int = 2010,
    band: int = 0,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,   # z.B. "Vulnerability_category" wenn vorhanden
    use_fixed_bounds_vuln: bool = True,     # NEU: Fixed bounds für Vulnerability
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:

    print("=== 5×5 PAPER-STYLE (WISSENSCHAFTLICH KORREKT) ===")
    print("GHG: Quintile via Quantile-Bins (rank -> qcut)")
    print("   → Jedes Quintile enthält ~20% der Länder (relative Verteilung)")
    if vuln_category_col and vuln_category_col in df_year.columns:
        print(f"Vulnerability: Kategorien aus '{vuln_category_col}' gemappt auf 1..5")
    else:
        print(f"Vulnerability: Fixed Bounds in 0.2er-Schritten (absolute Werte)")
        print(f"   1 = [0.0 - 0.2)  low")
        print(f"   2 = [0.2 - 0.4)  medium")
        print(f"   3 = [0.4 - 0.6)  high")
        print(f"   4 = [0.6 - 0.8)  severe")
        print(f"   5 = [0.8 - 1.0]  acute")
    print(f"Equitable band: ±{band}")
    print(f"Jahr: {year}\n")

    # --- Prep paper-style bins ---
    dfy = compute_bins_paper_style(
        df_year,
        ghg_col=ghg_col,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_vuln=use_fixed_bounds_vuln
    )
    dfy = add_equity_score_5x5(dfy, band=band)

    counts = build_count_matrix_5x5(dfy)
    score_grid = build_score_grid_5x5(band=band)

    # equity_score range in 5×5: -4..+4
    vmax, vmin = 4, -4
    cmap = make_blue_grey_yellow_cmap()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Layout ---
    fig, (ax_map, ax_mat) = plt.subplots(
        1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.8, 1]}
    )

    # --- A) Map ---
    world_merged.plot(
        column="equity_score",
        ax=ax_map,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.35,
        missing_kwds={"color": "lightgrey"},
        legend=False
    )
    ax_map.set_title(
        f"{year}: Equity gradient 5×5 (Paper-style bins, band = ±{band})",
        fontsize=14
    )
    ax_map.axis("off")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_map, fraction=0.035, pad=0.02)
    cbar.set_label("equity_score = GHG quintile − Vulnerability category/quintile\n(0 = equitable band)", fontsize=10)

    # --- B) Matrix ---
    ax_mat.imshow(
        score_grid,
        cmap=cmap,
        norm=norm,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal"
    )

    ax_mat.set_title("5×5 Matrix: Farbe = Equity, Zahl = # Länder", fontsize=14)
    ax_mat.set_xlabel("GHG per capita quintile (1 = low, 5 = high)")
    ax_mat.set_ylabel("Vulnerability (1 = low, 5 = acute)")
    ax_mat.set_xticks(np.arange(5))
    ax_mat.set_yticks(np.arange(5))
    ax_mat.set_xticklabels(range(1, 6))
    ax_mat.set_yticklabels(range(1, 6))
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    for i in range(5):
        for j in range(5):
            val = int(counts.iat[i, j])
            sc = int(score_grid[i, j])
            txt_color = "white" if (sc != 0 and abs(sc) >= 2) else "black"
            ax_mat.text(j, i, str(val), ha="center", va="center", color=txt_color, fontsize=11)

    ax_mat.text(-0.25, 4.1, "Forced\nrider side", ha="left", va="top", fontsize=9, color="black")
    ax_mat.text(3.8, -0.1, "Free\nrider side", ha="left", va="bottom", fontsize=9, color="black")
    ax_mat.text(2, 2, "Equitable" if band == 0 else "Equitable\nband",
                ha="center", va="center", fontsize=10, color="black")

    plt.tight_layout()
    plt.show()

    return counts, world_merged

In [ ]:
# WICHTIG: use_fixed_bounds_vuln=True setzen!
# Vulnerability wird in 0.2er-Schritten unterteilt (0.0-0.2, 0.2-0.4, 0.4-0.6, 0.6-0.8, 0.8-1.0)
counts_5x5_paper, world_merged_year = plot_map_and_matrix_5x5_equity_paper_style(
    df_year=merged_df_2010_mapped,   # dein Jahres-DF
    year=2010,                       # Jahr 2010 (weil merged_df_2010_mapped nur 2010-Daten hat)
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    vuln_category_col=None,          # Wir haben keine Kategorie-Spalte
    use_fixed_bounds_vuln=True       # Fixed bounds für wissenschaftliche Vergleichbarkeit
)

counts_5x5_paper

In [ ]:
# FUNKTIONEN FÜR JAHRESÜBERGREIFENDE VERGLEICHBARKEIT
def compute_global_ghg_quintile_bounds(df_all_years: pd.DataFrame, ghg_col: str = "GHG_per_capita") -> list:
    """
    Berechnet globale Quintile-Grenzen über ALLE Jahre hinweg.
    
    Returns:
        Liste mit 6 Grenzen [min, q1, q2, q3, q4, max] für 5 Quintile
    """
    all_ghg = df_all_years[ghg_col].dropna()
    
    # Quantile über alle Daten berechnen
    quantiles = all_ghg.quantile([0.0, 0.2, 0.4, 0.6, 0.8, 1.0]).tolist()
    
    print(f"=== Globale GHG Quintile-Grenzen (über alle Jahre) ===")
    print(f"Q1 (0-20%):  {quantiles[0]:.2f} - {quantiles[1]:.2f}")
    print(f"Q2 (20-40%): {quantiles[1]:.2f} - {quantiles[2]:.2f}")
    print(f"Q3 (40-60%): {quantiles[2]:.2f} - {quantiles[3]:.2f}")
    print(f"Q4 (60-80%): {quantiles[3]:.2f} - {quantiles[4]:.2f}")
    print(f"Q5 (80-100%): {quantiles[4]:.2f} - {quantiles[5]:.2f}")
    print()
    
    return quantiles


def assign_ghg_quintile_with_global_bounds(
    dfy: pd.DataFrame, 
    ghg_col: str, 
    global_bounds: list
) -> pd.Series:
    """
    Weist GHG-Werten Quintile basierend auf globalen Grenzen zu.
    
    Args:
        dfy: DataFrame für ein spezifisches Jahr
        ghg_col: Name der GHG-Spalte
        global_bounds: Liste mit [min, q1, q2, q3, q4, max] Grenzen
    
    Returns:
        Series mit Quintile-Werten 1-5
    """
    s = dfy[ghg_col]
    out = pd.Series(pd.NA, index=dfy.index, dtype="Int64")
    mask = s.notna()
    
    if mask.sum() == 0:
        return out
    
    # pd.cut mit globalen Grenzen
    bins = global_bounds
    labels = [1, 2, 3, 4, 5]
    
    cut_result = pd.cut(s[mask], bins=bins, labels=labels, include_lowest=True, duplicates='drop')
    out.loc[mask] = cut_result.astype(int)
    
    return out


def compute_bins_paper_style_with_global_bounds(
    dfy: pd.DataFrame,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_vuln: bool = True,
    global_ghg_bounds: list | None = None  # NEU: Globale GHG-Grenzen übergeben
) -> pd.DataFrame:
    """
    Paper-konform mit globalen Grenzen für Jahresvergleiche:
      - GHG: Quintile basierend auf globalen Grenzen (über alle Jahre) -> Jahresvergleichbar!
      - Vulnerability: Fixed bounds in 0.2er-Schritten (0-1) -> immer vergleichbar
    
    Args:
        global_ghg_bounds: Liste mit [min, q1, q2, q3, q4, max] wenn None, werden 
                          Quintile nur für dieses Jahr berechnet (wie vorher)
    """
    dfy = dfy.copy()
    
    # GHG Quintile
    if global_ghg_bounds is not None:
        # Nutze globale Grenzen für Jahresvergleichbarkeit
        dfy["ghg_quintile"] = assign_ghg_quintile_with_global_bounds(dfy, ghg_col, global_ghg_bounds)
    else:
        # Fallback: Quintile nur für dieses Jahr (wie vorher)
        dfy["ghg_quintile"] = add_quintiles_by_rank_qcut(dfy, ghg_col, n_bins=5)
    
    # Vulnerability Quintile (immer mit fixed bounds)
    dfy["vuln_quintile"] = to_vuln_ordinal_1to5(
        dfy,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_0to1=use_fixed_bounds_vuln
    )
    
    return dfy


def plot_map_and_matrix_5x5_equity_comparable_years(
    df_year: pd.DataFrame,
    year: int = 2010,
    band: int = 0,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_vuln: bool = True,
    global_ghg_bounds: list | None = None,  # NEU: Globale GHG-Grenzen
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:
    """
    Wie plot_map_and_matrix_5x5_equity_paper_style_v2 aber mit Unterstützung für
    globale GHG-Grenzen für Jahresvergleiche.
    """

    print("=== 5×5 JAHRESVERGLEICHBARE QUINTILE ===")
    if global_ghg_bounds is not None:
        print("GHG: Quintile via GLOBALE Grenzen (über alle Jahre)")
        print("   → Quintile sind über Jahre vergleichbar!")
    else:
        print("GHG: Quintile via Quantile-Bins nur für dieses Jahr (rank -> qcut)")
        print("   → Jedes Quintile enthält ~20% der Länder dieses Jahres")
    
    print(f"Vulnerability: Fixed Bounds in 0.2er-Schritten (immer vergleichbar)")
    print(f"   1 = [0.0 - 0.2)  low")
    print(f"   2 = [0.2 - 0.4)  medium")
    print(f"   3 = [0.4 - 0.6)  high")
    print(f"   4 = [0.6 - 0.8)  severe")
    print(f"   5 = [0.8 - 1.0]  acute")
    print(f"Equitable band: ±{band}")
    print(f"Jahr: {year}\n")

    # --- Prep bins mit globalen Grenzen ---
    dfy = compute_bins_paper_style_with_global_bounds(
        df_year,
        ghg_col=ghg_col,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_vuln=use_fixed_bounds_vuln,
        global_ghg_bounds=global_ghg_bounds
    )
    dfy = add_equity_score_5x5(dfy, band=band)

    counts = build_count_matrix_5x5(dfy)
    score_grid = build_score_grid_5x5(band=band)

    # equity_score range in 5×5: -4..+4
    vmax, vmin = 4, -4
    cmap = make_blue_grey_yellow_cmap()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Layout ---
    fig, (ax_map, ax_mat) = plt.subplots(
        1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [2.2, 1]}, dpi=300
    )

    # --- A) Map ---
    # Länder mit Daten normal plotten
    world_merged[world_merged["equity_score"].notna()].plot(
        column="equity_score",
        ax=ax_map,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.35,
        legend=False
    )
    
    # Länder ohne Daten mit Schraffierung (hatching)
    world_merged[world_merged["equity_score"].isna()].plot(
        ax=ax_map,
        color="lightgrey",
        edgecolor="black",
        linewidth=0.35,
        hatch="///",
        legend=False
    )
    
    bounds_label = "global bounds" if global_ghg_bounds else "year-specific"
    # ax_map.set_title(
    #     f"{year}: Equity gradient 5×5 (GHG: {bounds_label}, Vuln: fixed bounds, band = ±{band})",
    #     fontsize=14
    # )
    ax_map.set_title(
        f"{year}: GHG-Emissions vs Vulnerability Equity Map",
        fontsize=15
    )
    ax_map.axis("off")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax_map, fraction=0.025, pad=-0.02)
    cbar.set_label("equity_score = GHG quintile − Vulnerability quintile\n(0 = equitable band)", fontsize=10)
    
    # Legende für fehlende Daten
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightgrey', edgecolor='black', hatch='///', label='Keine Daten')
    ]
    ax_map.legend(handles=legend_elements, loc='lower left', fontsize=9, bbox_to_anchor=(0.0, -0.05))

    # --- B) Matrix ---
    ax_mat.imshow(
        score_grid,
        cmap=cmap,
        norm=norm,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal"
    )

    ax_mat.set_title("Equity Matrix\n(Farbe = Equity, Zahl = # Länder) ", fontsize=14)
    ax_mat.set_xlabel("GHG per capita quintile (1 = low, 5 = high)")
    ax_mat.set_ylabel("Vulnerability (1 = low, 5 = acute)")
    ax_mat.set_xticks(np.arange(5))
    ax_mat.set_yticks(np.arange(5))
    ax_mat.set_xticklabels(range(1, 6))
    ax_mat.set_yticklabels(range(1, 6))
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    for i in range(5):
        for j in range(5):
            val = int(counts.iat[i, j])
            sc = int(score_grid[i, j])
            txt_color = "black"
            ax_mat.text(j, i, str(val), ha="center", va="center", color=txt_color, fontsize=11)

    ax_mat.text(-0.45, 4.45, "Forced\nrider side", ha="left", va="top", fontsize=9, color="black")
    ax_mat.text(3.85, -0.45, "Free\nrider side", ha="left", va="bottom", fontsize=9, color="black")
    ax_mat.text(1.8, 2.25, "Equitable" if band == 0 else "Equitable\nband",
                ha="center", va="center", rotation=45, fontsize=9, color="black")

    plt.tight_layout()
    plt.show()

    return counts, world_merged

In [ ]:
def prepare_year_data(year: int, df_source: pd.DataFrame = None, path: str = "data/merged_df.csv") -> pd.DataFrame:
    """
    Filtert Daten nach Jahr und wendet ISO3-Mappings an.
    
    Args:
        year: Jahreszahl (z.B. 2010, 2023)
        df_source: Optional - bereits geladenes DataFrame. Wenn None, wird von path geladen
        path: Pfad zur CSV-Datei (nur relevant wenn df_source None ist)
    
    Returns:
        DataFrame gefiltert nach Jahr mit angewendeten ISO3-Mappings
    """
    # Lade Daten falls nicht übergeben
    if df_source is None:
        df = pd.read_csv(path)
    else:
        df = df_source.copy()
    
    # Filtere nach Jahr
    df_year = df[df['Year'] == year].copy()
    
    if len(df_year) == 0:
        print(f"⚠️ WARNUNG: Keine Daten für Jahr {year} gefunden!")
        return df_year
    
    # ISO3 Mappings (wie in merged_df_2010_mapped)
    iso3_mappings = {
        'SCG': ['MNE', 'SRB'],  # Serbia and Montenegro -> Montenegro and Serbia
        'ISR': ['PSE'],         # Israel -> Palestine
        'SDN': ['SSD'],         # Sudan -> South Sudan
        'CHN': ['TWN']          # China -> Taiwan
    }
    
    # Wende Mappings an
    additional_rows = []
    for source_iso3, target_iso3_list in iso3_mappings.items():
        source_data = df_year[df_year['ISO3'] == source_iso3]
        if not source_data.empty:
            for target_iso3 in target_iso3_list:
                new_rows = source_data.copy()
                new_rows['ISO3'] = target_iso3
                additional_rows.append(new_rows)
    
    # Füge zusätzliche Zeilen hinzu
    if additional_rows:
        df_year_mapped = pd.concat([df_year] + additional_rows, ignore_index=True)
    else:
        df_year_mapped = df_year
    
    print(f"✓ Jahr {year} vorbereitet:")
    print(f"  - Original Zeilen: {len(df_year)}")
    print(f"  - Nach Mapping: {len(df_year_mapped)}")
    print(f"  - GHG Range: {df_year_mapped['GHG_per_capita'].min():.2f} - {df_year_mapped['GHG_per_capita'].max():.2f}")
    print(f"  - Vulnerability Range: {df_year_mapped['Vulnerability'].min():.3f} - {df_year_mapped['Vulnerability'].max():.3f}")
    
    return df_year_mapped

In [ ]:
merged_df_2010_mapped = prepare_year_data(2010)
merged_df_2023_mapped = prepare_year_data(2023)
merged_df_2015_mapped = prepare_year_data(2015)
merged_df_1995_mapped = prepare_year_data(1995)

In [ ]:
global_ghg_bounds = compute_global_ghg_quintile_bounds(
    df_all_years=merged_df,  # Alle Daten (alle Jahre)
    ghg_col="GHG_per_capita"
)

counts_1995_global, world_1995_global = plot_map_and_matrix_5x5_equity_comparable_years(
    df_year=merged_df_1995_mapped,
    year=1995,
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    vuln_category_col=None,
    use_fixed_bounds_vuln=True,
    global_ghg_bounds=global_ghg_bounds  # <- Nutzt globale Grenzen!
)

print("\nLänder-Verteilung 1995 (mit globalen Grenzen):")
print(counts_1995_global)

In [ ]:
counts_2010_global, world_2010_global = plot_map_and_matrix_5x5_equity_comparable_years(
    df_year=merged_df_2010_mapped,
    year=2010,
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    vuln_category_col=None,
    use_fixed_bounds_vuln=True,
    global_ghg_bounds=global_ghg_bounds  # <- Nutzt globale Grenzen!
)

print("\nLänder-Verteilung 2010 (mit globalen Grenzen):")
print(counts_2010_global)

In [ ]:
counts_2015_global, world_2015_global = plot_map_and_matrix_5x5_equity_comparable_years(
    df_year=merged_df_2015_mapped,
    year=2015,
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    vuln_category_col=None,
    use_fixed_bounds_vuln=True,
    global_ghg_bounds=global_ghg_bounds  # <- Nutzt globale Grenzen!
)

print("\nLänder-Verteilung 2015 (mit globalen Grenzen):")
print(counts_2015_global)

In [ ]:
counts_2023_global, world_2023_global = plot_map_and_matrix_5x5_equity_comparable_years(
    df_year=merged_df_2023_mapped,
    year=2023,
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    vuln_category_col=None,
    use_fixed_bounds_vuln=True,
    global_ghg_bounds=global_ghg_bounds  # <- Nutzt globale Grenzen!
)

print("\nLänder-Verteilung 2023 (mit globalen Grenzen):")
print(counts_2023_global)

In [ ]:
def plot_map_with_inset_matrix_5x5(
    df_year: pd.DataFrame,
    year: int = 2010,
    band: int = 0,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_vuln: bool = True,
    global_ghg_bounds: list | None = None,
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:
    """
    Wie plot_map_and_matrix_5x5_equity_comparable_years aber mit Matrix als Inset
    links unten auf der Karte (keine Colorbar).
    """

    print("=== 5×5 JAHRESVERGLEICHBARE QUINTILE (INSET-MATRIX) ===")
    if global_ghg_bounds is not None:
        print("GHG: Quintile via GLOBALE Grenzen (über alle Jahre)")
        print("   → Quintile sind über Jahre vergleichbar!")
    else:
        print("GHG: Quintile via Quantile-Bins nur für dieses Jahr (rank -> qcut)")
        print("   → Jedes Quintile enthält ~20% der Länder dieses Jahres")
    
    print(f"Vulnerability: Fixed Bounds in 0.2er-Schritten (immer vergleichbar)")
    print(f"Equitable band: ±{band}")
    print(f"Jahr: {year}\n")

    # --- Prep bins mit globalen Grenzen ---
    dfy = compute_bins_paper_style_with_global_bounds(
        df_year,
        ghg_col=ghg_col,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_vuln=use_fixed_bounds_vuln,
        global_ghg_bounds=global_ghg_bounds
    )
    dfy = add_equity_score_5x5(dfy, band=band)

    counts = build_count_matrix_5x5(dfy)
    score_grid = build_score_grid_5x5(band=band)

    # equity_score range in 5×5: -4..+4
    vmax, vmin = 4, -4
    cmap = make_blue_grey_yellow_cmap()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Layout: Nur eine Figure mit einem Hauptplot ---
    # fig, ax_map = plt.subplots(figsize=(16, 9), dpi=600)

    # A0 Poster Abmessungen
    fig, ax_map = plt.subplots(figsize=(16, 9))  # dpi weglassen für Vektor

    # --- Map ---
    # Länder mit Daten normal plotten
    world_merged[world_merged["equity_score"].notna()].plot(
        column="equity_score",
        ax=ax_map,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.35,
        legend=False
    )
    
    # Länder ohne Daten mit Schraffierung (hatching)
    world_merged[world_merged["equity_score"].isna()].plot(
        ax=ax_map,
        color="lightgrey",
        edgecolor="black",
        linewidth=0.35,
        hatch="///",
        legend=False
    )
    
    bounds_label = "global bounds" if global_ghg_bounds else "year-specific"
    ax_map.set_title(
        f"{year}: Emissionen vs Verwundbarkeit",
        fontsize=15,
        y=0.96
    )
    ax_map.axis("off")
    
    # Legende für fehlende Daten
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightgrey', edgecolor='black', hatch='///', label='Keine Daten')
    ]
    ax_map.legend(handles=legend_elements, loc='lower left', fontsize=9, bbox_to_anchor=(0.85, 0.18))

    # --- Inset für Matrix links unten ---
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    ax_inset = inset_axes(ax_map, width="32%", height="32%", loc='lower left', 
                          bbox_to_anchor=(0.02, 0.2, 1, 1), bbox_transform=ax_map.transAxes,
                          borderpad=0)
    
    # Matrix im Inset plotten (transponiert)
    ax_inset.imshow(
        score_grid.T,
        cmap=cmap,
        norm=norm,
        origin="lower",
        interpolation="none",
        aspect="equal"
    )

    # ax_inset.set_title("Equity Matrix", fontsize=10, pad=5)
    ax_inset.set_xlabel("Verwundbarkeit", fontsize=11)
    ax_inset.set_ylabel("Emissionen", fontsize=11)
    ax_inset.set_xticks(np.arange(5))
    ax_inset.set_yticks(np.arange(5))
    ax_inset.set_xticklabels(range(1, 6), fontsize=7)
    ax_inset.set_yticklabels(range(1, 6), fontsize=7)
    ax_inset.grid(False)
    ax_inset.tick_params(labelsize=7)

    # # Zahlen in Matrix
    # for i in range(5):
    #     for j in range(5):
    #         val = int(counts.iat[i, j])
    #         ax_inset.text(j, i, str(val), ha="center", va="center", 
    #                      color="black", fontsize=7, fontweight='bold')

    plt.tight_layout()
    plt.show()

    # Plot speichern
    out_base = f"equity_map_{year}_band{band}"

    fig.savefig(out_base + ".svg", transparent=False, bbox_inches="tight", pad_inches=0.02)
    # fig.savefig(out_base + ".pdf", bbox_inches="tight", pad_inches=0.02)  # optional

    return counts, world_merged

In [ ]:
# Alternative inklusive Farbschemata für die Equity Matrix
# Alle sind optimiert für Farbenblindheit (Deuteranopie, Protanopie)

def make_viridis_diverging_cmap() -> LinearSegmentedColormap:
    """
    Viridis-basiertes divergierendes Schema: Violett → Grau → Gelb-Grün
    - Sehr gut für alle Arten von Farbenblindheit
    - Wissenschaftlich validiert (perceptually uniform)
    """
    return LinearSegmentedColormap.from_list(
        "viridis_diverging",
        ["#440154", "#F2F2F2", "#FDE724"],  # violett, grau, gelb-grün
        N=256
    )

def make_brown_teal_cmap() -> LinearSegmentedColormap:
    """
    Braun → Weiß → Türkis (ColorBrewer BrBG)
    - Exzellent für Deuteranopie und Protanopie
    - Natürliche Assoziation: trocken/kühl
    """
    return LinearSegmentedColormap.from_list(
        "brown_teal",
        ["#8C510A", "#F5F5F5", "#01665E"],  # braun, weiß, türkis
        N=256
    )

def make_orange_purple_cmap() -> LinearSegmentedColormap:
    """
    Orange → Weiß → Lila (ColorBrewer PuOr)
    - Sehr gut für alle Farbenblindheitstypen
    - Hoher Kontrast, gut lesbar
    """
    return LinearSegmentedColormap.from_list(
        "orange_purple",
        ["#E66101", "#F7F7F7", "#5E3C99"],  # orange, weiß, lila
        N=256
    )

def make_pink_green_cmap() -> LinearSegmentedColormap:
    """
    Pink → Grau → Grün (ColorBrewer PiYG)
    - Gut für Deuteranopie
    - ACHTUNG: Weniger gut für Protanopie (Rot-Schwäche)
    """
    return LinearSegmentedColormap.from_list(
        "pink_green",
        ["#C51B7D", "#F7F7F7", "#4D9221"],  # pink, weiß, grün
        N=256
    )

def make_sunset_cmap() -> LinearSegmentedColormap:
    """
    Blau → Grau → Orange (wissenschaftlich optimiert)
    - Ausgezeichnet für alle Farbenblindheitstypen
    - Hoher Kontrast, natürliche Assoziation
    """
    return LinearSegmentedColormap.from_list(
        "sunset",
        ["#3B4CC0", "#DDDDDD", "#F77E5E"],  # dunkelblau, grau, orange
        N=256
    )

# Aktuelle Colormap (zum Vergleich)
def make_blue_grey_yellow_cmap() -> LinearSegmentedColormap:
    """
    Aktuell: Blau → Grau → Gelb
    - Gut für Farbenblindheit
    """
    return LinearSegmentedColormap.from_list(
        "blue_grey_yellow",
        ["#2C7BB6", "#F2F2F2", "#FEE08B"],  # blau, grau, gelb
        N=256
    )

# Visualisierung aller Farbschemata zum Vergleich
fig, axes = plt.subplots(6, 1, figsize=(12, 8))
gradient = np.linspace(-4, 4, 256).reshape(1, -1)

cmaps = [
    ("Aktuell: Blau-Grau-Gelb", make_blue_grey_yellow_cmap()),
    ("Viridis: Violett-Grau-Gelb-Grün", make_viridis_diverging_cmap()),
    ("Braun-Türkis (BrBG)", make_brown_teal_cmap()),
    ("Orange-Lila (PuOr)", make_orange_purple_cmap()),
    ("Pink-Grün (PiYG)", make_pink_green_cmap()),
    ("Sunset: Blau-Grau-Orange", make_sunset_cmap()),
]

norm = TwoSlopeNorm(vmin=-4, vcenter=0, vmax=4)

for ax, (name, cmap) in zip(axes, cmaps):
    ax.imshow(gradient, aspect='auto', cmap=cmap, norm=norm)
    ax.set_yticks([])
    ax.set_xticks([-4, -2, 0, 2, 4])
    ax.set_ylabel(name, fontsize=10)
    if ax == axes[-1]:
        ax.set_xlabel("Equity Score", fontsize=10)

plt.suptitle("Inklusive Farbschemata für Equity Matrix\n(alle optimiert für Farbenblindheit)", 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nEmpfehlungen:")
print("1. Braun-Türkis: Beste Wahl für maximale Inklusivität")
print("2. Sunset (Blau-Orange): Sehr gut, hoher Kontrast")
print("3. Viridis: Wissenschaftlich validiert, perceptually uniform")
print("4. Orange-Lila: Hoher Kontrast, gut lesbar")
print("\nZum Ändern: Ersetze 'make_blue_grey_yellow_cmap()' durch eine der Funktionen oben")

In [ ]:
# Test: 2010 mit Matrix als Inset links unten
print("="*60)
print("JAHR 2023 mit Inset-Matrix (keine separate Colorbar)")
print("="*60)
counts_2023_inset, world_2023_inset = plot_map_with_inset_matrix_5x5(
    df_year=merged_df_2023_mapped,
    year=2023,
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    vuln_category_col=None,
    use_fixed_bounds_vuln=True,
    global_ghg_bounds=global_ghg_bounds
)

In [ ]:
# FUNKTIONEN FÜR JAHRESÜBERGREIFENDE VERGLEICHBARKEIT
def compute_global_ghg_quintile_bounds(df_all_years: pd.DataFrame, ghg_col: str = "GHG_per_capita") -> list:
    """
    Berechnet globale Quintile-Grenzen über ALLE Jahre hinweg.
    
    Returns:
        Liste mit 6 Grenzen [min, q1, q2, q3, q4, max] für 5 Quintile
    """
    all_ghg = df_all_years[ghg_col].dropna()
    
    # Quantile über alle Daten berechnen
    quantiles = all_ghg.quantile([0.0, 0.2, 0.4, 0.6, 0.8, 1.0]).tolist()
    
    print(f"=== Globale GHG Quintile-Grenzen (über alle Jahre) ===")
    print(f"Q1 (0-20%):  {quantiles[0]:.2f} - {quantiles[1]:.2f}")
    print(f"Q2 (20-40%): {quantiles[1]:.2f} - {quantiles[2]:.2f}")
    print(f"Q3 (40-60%): {quantiles[2]:.2f} - {quantiles[3]:.2f}")
    print(f"Q4 (60-80%): {quantiles[3]:.2f} - {quantiles[4]:.2f}")
    print(f"Q5 (80-100%): {quantiles[4]:.2f} - {quantiles[5]:.2f}")
    print()
    
    return quantiles


def assign_ghg_quintile_with_global_bounds(
    dfy: pd.DataFrame, 
    ghg_col: str, 
    global_bounds: list
) -> pd.Series:
    """
    Weist GHG-Werten Quintile basierend auf globalen Grenzen zu.
    
    Args:
        dfy: DataFrame für ein spezifisches Jahr
        ghg_col: Name der GHG-Spalte
        global_bounds: Liste mit [min, q1, q2, q3, q4, max] Grenzen
    
    Returns:
        Series mit Quintile-Werten 1-5
    """
    s = dfy[ghg_col]
    out = pd.Series(pd.NA, index=dfy.index, dtype="Int64")
    mask = s.notna()
    
    if mask.sum() == 0:
        return out
    
    # pd.cut mit globalen Grenzen
    bins = global_bounds
    labels = [1, 2, 3, 4, 5]
    
    cut_result = pd.cut(s[mask], bins=bins, labels=labels, include_lowest=True, duplicates='drop')
    out.loc[mask] = cut_result.astype(int)
    
    return out


def compute_bins_paper_style_with_global_bounds(
    dfy: pd.DataFrame,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_vuln: bool = True,
    global_ghg_bounds: list | None = None  # NEU: Globale GHG-Grenzen übergeben
) -> pd.DataFrame:
    """
    Paper-konform mit globalen Grenzen für Jahresvergleiche:
      - GHG: Quintile basierend auf globalen Grenzen (über alle Jahre) -> Jahresvergleichbar!
      - Vulnerability: Fixed bounds in 0.2er-Schritten (0-1) -> immer vergleichbar
    
    Args:
        global_ghg_bounds: Liste mit [min, q1, q2, q3, q4, max] wenn None, werden 
                          Quintile nur für dieses Jahr berechnet (wie vorher)
    """
    dfy = dfy.copy()
    
    # GHG Quintile
    if global_ghg_bounds is not None:
        # Nutze globale Grenzen für Jahresvergleichbarkeit
        dfy["ghg_quintile"] = assign_ghg_quintile_with_global_bounds(dfy, ghg_col, global_ghg_bounds)
    else:
        # Fallback: Quintile nur für dieses Jahr (wie vorher)
        dfy["ghg_quintile"] = add_quintiles_by_rank_qcut(dfy, ghg_col, n_bins=5)
    
    # Vulnerability Quintile (immer mit fixed bounds)
    dfy["vuln_quintile"] = to_vuln_ordinal_1to5(
        dfy,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_0to1=use_fixed_bounds_vuln
    )
    
    return dfy


def plot_map_and_matrix_5x5_equity_comparable_years(
    df_year: pd.DataFrame,
    year: int = 2010,
    band: int = 0,
    ghg_col: str = "GHG_per_capita",
    vuln_col: str = "Vulnerability",
    vuln_category_col: str | None = None,
    use_fixed_bounds_vuln: bool = True,
    global_ghg_bounds: list | None = None,  # NEU: Globale GHG-Grenzen
    world_url: str = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip",
) -> tuple[pd.DataFrame, gpd.GeoDataFrame]:
    """
    Wie plot_map_and_matrix_5x5_equity_paper_style_v2 aber mit Unterstützung für
    globale GHG-Grenzen für Jahresvergleiche.
    """

    print("=== 5×5 JAHRESVERGLEICHBARE QUINTILE ===")
    if global_ghg_bounds is not None:
        print("GHG: Quintile via GLOBALE Grenzen (über alle Jahre)")
        print("   → Quintile sind über Jahre vergleichbar!")
    else:
        print("GHG: Quintile via Quantile-Bins nur für dieses Jahr (rank -> qcut)")
        print("   → Jedes Quintile enthält ~20% der Länder dieses Jahres")
    
    print(f"Vulnerability: Fixed Bounds in 0.2er-Schritten (immer vergleichbar)")
    print(f"   1 = [0.0 - 0.2)  low")
    print(f"   2 = [0.2 - 0.4)  medium")
    print(f"   3 = [0.4 - 0.6)  high")
    print(f"   4 = [0.6 - 0.8)  severe")
    print(f"   5 = [0.8 - 1.0]  acute")
    print(f"Equitable band: ±{band}")
    print(f"Jahr: {year}\n")

    # --- Prep bins mit globalen Grenzen ---
    dfy = compute_bins_paper_style_with_global_bounds(
        df_year,
        ghg_col=ghg_col,
        vuln_col=vuln_col,
        vuln_category_col=vuln_category_col,
        use_fixed_bounds_vuln=use_fixed_bounds_vuln,
        global_ghg_bounds=global_ghg_bounds
    )
    dfy = add_equity_score_5x5(dfy, band=band)

    counts = build_count_matrix_5x5(dfy)
    score_grid = build_score_grid_5x5(band=band)

    # equity_score range in 5×5: -4..+4
    vmax, vmin = 4, -4
    cmap = make_blue_grey_yellow_cmap()
    norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

    # --- Merge geometry ---
    world = gpd.read_file(world_url)
    world_merged = world.merge(dfy, left_on="ISO_A3_EH", right_on="ISO3", how="left")

    # --- Layout ---
    fig, (ax_map, ax_mat) = plt.subplots(
        1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [2.2, 1]}, dpi=300
    )

    # --- A) Map ---
    # Länder mit Daten normal plotten
    world_merged[world_merged["equity_score"].notna()].plot(
        column="equity_score",
        ax=ax_map,
        cmap=cmap,
        norm=norm,
        edgecolor="black",
        linewidth=0.35,
        legend=False
    )
    
    # Länder ohne Daten mit Schraffierung (hatching)
    world_merged[world_merged["equity_score"].isna()].plot(
        ax=ax_map,
        color="lightgrey",
        edgecolor="black",
        linewidth=0.35,
        hatch="///",
        legend=False
    )
    
    bounds_label = "global bounds" if global_ghg_bounds else "year-specific"
    # ax_map.set_title(
    #     f"{year}: Equity gradient 5×5 (GHG: {bounds_label}, Vuln: fixed bounds, band = ±{band})",
    #     fontsize=14
    # )
    ax_map.set_title(
        f"{year}: GHG-Emissions vs Vulnerability Equity Map",
        fontsize=15,
        y=0.98
    )
    ax_map.axis("off")

    # sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    # sm.set_array([])
    # cbar = fig.colorbar(sm, ax=ax_map, fraction=0.025, pad=-0.02)
    # cbar.set_label("equity_score = GHG quintile − Vulnerability quintile\n(0 = equitable band)", fontsize=10)
    
    # Legende für fehlende Daten
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='lightgrey', edgecolor='black', hatch='///', label='Keine Daten')
    ]
    ax_map.legend(handles=legend_elements, loc='lower left', fontsize=9, bbox_to_anchor=(0.05, 0.15))

    # --- B) Matrix ---
    ax_mat.imshow(
        score_grid,
        cmap=cmap,
        norm=norm,
        origin="lower",
        interpolation="none",
        resample=False,
        aspect="equal"
    )

    ax_mat.set_title("Equity Matrix\n(Farbe = Equity, Zahl = # Länder) ", fontsize=14)
    ax_mat.set_xlabel("GHG per capita quintile (1 = low, 5 = high)")
    ax_mat.set_ylabel("Vulnerability (1 = low, 5 = acute)")
    ax_mat.set_xticks(np.arange(5))
    ax_mat.set_yticks(np.arange(5))
    ax_mat.set_xticklabels(range(1, 6))
    ax_mat.set_yticklabels(range(1, 6))
    ax_mat.grid(False)
    ax_mat.minorticks_off()

    for i in range(5):
        for j in range(5):
            val = int(counts.iat[i, j])
            sc = int(score_grid[i, j])
            txt_color = "black"
            ax_mat.text(j, i, str(val), ha="center", va="center", color=txt_color, fontsize=11)

    ax_mat.text(-0.45, 4.45, "Forced\nrider side", ha="left", va="top", fontsize=9, color="black")
    ax_mat.text(3.85, -0.45, "Free\nrider side", ha="left", va="bottom", fontsize=9, color="black")
    ax_mat.text(1.8, 2.25, "Equitable" if band == 0 else "Equitable\nband",
                ha="center", va="center", rotation=45, fontsize=9, color="black")

    plt.tight_layout()
    plt.show()

    return counts, world_merged

In [ ]:
# Plot für 2010 ohne Colorbar
counts, world_merged = plot_map_and_matrix_5x5_equity_comparable_years(
    df_year=merged_df_2010_mapped,
    year=2010,
    band=0,
    ghg_col="GHG_per_capita",
    vuln_col="Vulnerability",
    use_fixed_bounds_vuln=True,
    global_ghg_bounds=global_ghg_bounds,
    world_url="https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
)